## Preliminary

In [5]:
import polars as pl
from pathlib import Path

import sys


ROOT = Path.cwd().parents[0]

sys.path.append(str(ROOT / "src"))

from fcts import inspect_lazy


DATA_PATH = ROOT / "data"

pulled_data = DATA_PATH / "daioe_scb_years_all_levels.parquet"

processed_data = DATA_PATH / "processed_data.parquet"

## 1. Read the Data

In [6]:
pulled_lf = pl.scan_parquet(pulled_data) 

In [7]:
pulled_lf.collect_schema()

Schema([('level', String),
        ('ssyk_code', String),
        ('age', String),
        ('sex', String),
        ('year', Int64),
        ('count', Int64),
        ('occupation', String),
        ('age_group', String),
        ('weight_sum', Int64),
        ('daioe_allapps_avg', Float64),
        ('daioe_stratgames_avg', Float64),
        ('daioe_videogames_avg', Float64),
        ('daioe_imgrec_avg', Float64),
        ('daioe_imgcompr_avg', Float64),
        ('daioe_imggen_avg', Float64),
        ('daioe_readcompr_avg', Float64),
        ('daioe_lngmod_avg', Float64),
        ('daioe_translat_avg', Float64),
        ('daioe_speechrec_avg', Float64),
        ('daioe_genai_avg', Float64),
        ('daioe_allapps_wavg', Float64),
        ('daioe_stratgames_wavg', Float64),
        ('daioe_videogames_wavg', Float64),
        ('daioe_imgrec_wavg', Float64),
        ('daioe_imgcompr_wavg', Float64),
        ('daioe_imggen_wavg', Float64),
        ('daioe_readcompr_wavg', Float64),
      

## 2. Compute the 1-Yr, 3-Yr and 5-Yr Change

Aggregate to unemployment totals by `level`, `ssyk_code`, `age`, `sex`, and `year`, then compute absolute and percent changes over 1, 3, and 5 years using window shifts.

In [8]:
keys = ["level", "ssyk_code", "age", "sex"] 
val = "count"

pulled_with_changes = (
    pulled_lf 
    .group_by([*keys, "year"])
    .agg(pl.col(val).sum().alias("unemp_count"))
    .sort([*keys, "year"])
    .with_columns([
        (pl.col("unemp_count") - pl.col("unemp_count").shift(1).over(keys)).alias("chg_1y"),
        (pl.col("unemp_count") - pl.col("unemp_count").shift(3).over(keys)).alias("chg_3y"),
        (pl.col("unemp_count") - pl.col("unemp_count").shift(5).over(keys)).alias("chg_5y"),

        ((pl.col("unemp_count") / pl.col("unemp_count").shift(1).over(keys) - 1) * 100).alias("pct_chg_1y"),
        ((pl.col("unemp_count") / pl.col("unemp_count").shift(3).over(keys) - 1) * 100).alias("pct_chg_3y"),
        ((pl.col("unemp_count") / pl.col("unemp_count").shift(5).over(keys) - 1) * 100).alias("pct_chg_5y"),
    ])
)

pulled_with_changes.head(20).collect()


level,ssyk_code,age,sex,year,unemp_count,chg_1y,chg_3y,chg_5y,pct_chg_1y,pct_chg_3y,pct_chg_5y
str,str,str,str,i64,i64,i64,i64,i64,f64,f64,f64
"""SSYK1""","""1""","""065-69""","""men""",2014,0,null,null,null,null,null,null
"""SSYK1""","""1""","""065-69""","""men""",2015,0,0,null,null,NaN,null,null
"""SSYK1""","""1""","""065-69""","""men""",2016,0,0,null,null,NaN,null,null
"""SSYK1""","""1""","""065-69""","""men""",2017,0,0,0,null,NaN,NaN,null
"""SSYK1""","""1""","""065-69""","""men""",2018,0,0,0,null,NaN,NaN,null
…,…,…,…,…,…,…,…,…,…,…,…
"""SSYK1""","""1""","""065-69""","""women""",2018,0,0,0,null,NaN,NaN,null
"""SSYK1""","""1""","""065-69""","""women""",2019,0,0,0,0,NaN,NaN,NaN
"""SSYK1""","""1""","""065-69""","""women""",2020,0,0,0,0,NaN,NaN,NaN


In [9]:
inspect_lazy(pulled_with_changes)

Rows: 137,060
Columns: 12


## 3. Merge and write to processed data




Join the engineered change metrics back onto the original table with a left join on the key dimensions, so all original rows are preserved.

In [10]:
join_col = ["year", "level", "ssyk_code", "age", "sex"]

processed_lf = pulled_lf\
    .join(
        pulled_with_changes,
        left_on = join_col,
        right_on = join_col,
        how = "left"
    )

In [11]:
#inspect_lazy(processed_lf)
processed_lf.collect_schema()

Schema([('level', String),
        ('ssyk_code', String),
        ('age', String),
        ('sex', String),
        ('year', Int64),
        ('count', Int64),
        ('occupation', String),
        ('age_group', String),
        ('weight_sum', Int64),
        ('daioe_allapps_avg', Float64),
        ('daioe_stratgames_avg', Float64),
        ('daioe_videogames_avg', Float64),
        ('daioe_imgrec_avg', Float64),
        ('daioe_imgcompr_avg', Float64),
        ('daioe_imggen_avg', Float64),
        ('daioe_readcompr_avg', Float64),
        ('daioe_lngmod_avg', Float64),
        ('daioe_translat_avg', Float64),
        ('daioe_speechrec_avg', Float64),
        ('daioe_genai_avg', Float64),
        ('daioe_allapps_wavg', Float64),
        ('daioe_stratgames_wavg', Float64),
        ('daioe_videogames_wavg', Float64),
        ('daioe_imgrec_wavg', Float64),
        ('daioe_imgcompr_wavg', Float64),
        ('daioe_imggen_wavg', Float64),
        ('daioe_readcompr_wavg', Float64),
      

### 3.1 Reorder and Validate Columns

Bring key analytical columns to the front and keep all remaining columns afterward.

A schema snapshot is used to build the final order while keeping all remaining columns.

In [12]:
first_cols = [
    "year",
    "level",
    "ssyk_code",
    "occupation",
    "sex",
    "age",
    "age_group",
    "count",
    "weight_sum",
    "unemp_count",
    "chg_1y",
    "chg_3y",
    "chg_5y",
    "pct_chg_1y",
    "pct_chg_3y",
    "pct_chg_5y"
]

schema_cols = list(processed_lf.collect_schema())

missing_first_cols = [col for col in first_cols if col not in schema_cols]

other_cols = [col for col in schema_cols if col not in first_cols]

processed_lf_reorder_cols =  processed_lf\
    .select(
    first_cols + other_cols
    )\
        .sort(join_col, nulls_last = True, descending = True)

processed_lf_reorder_cols.limit(10).collect()

year,level,ssyk_code,occupation,sex,age,age_group,count,weight_sum,unemp_count,chg_1y,chg_3y,chg_5y,pct_chg_1y,pct_chg_3y,pct_chg_5y,daioe_allapps_avg,daioe_stratgames_avg,daioe_videogames_avg,daioe_imgrec_avg,daioe_imgcompr_avg,daioe_imggen_avg,daioe_readcompr_avg,daioe_lngmod_avg,daioe_translat_avg,daioe_speechrec_avg,daioe_genai_avg,daioe_allapps_wavg,daioe_stratgames_wavg,daioe_videogames_wavg,daioe_imgrec_wavg,daioe_imgcompr_wavg,daioe_imggen_wavg,daioe_readcompr_wavg,daioe_lngmod_wavg,daioe_translat_wavg,daioe_speechrec_wavg,daioe_genai_wavg,level_right,pctl_daioe_allapps_avg,pctl_daioe_stratgames_avg,pctl_daioe_videogames_avg,pctl_daioe_imgrec_avg,pctl_daioe_imgcompr_avg,pctl_daioe_imggen_avg,pctl_daioe_readcompr_avg,pctl_daioe_lngmod_avg,pctl_daioe_translat_avg,pctl_daioe_speechrec_avg,pctl_daioe_genai_avg,pctl_daioe_allapps_wavg,pctl_daioe_stratgames_wavg,pctl_daioe_videogames_wavg,pctl_daioe_imgrec_wavg,pctl_daioe_imgcompr_wavg,pctl_daioe_imggen_wavg,pctl_daioe_readcompr_wavg,pctl_daioe_lngmod_wavg,pctl_daioe_translat_wavg,pctl_daioe_speechrec_wavg,pctl_daioe_genai_wavg,daioe_allapps_Level_Exposure,daioe_stratgames_Level_Exposure,daioe_videogames_Level_Exposure,daioe_imgrec_Level_Exposure,daioe_imgcompr_Level_Exposure,daioe_imggen_Level_Exposure,daioe_readcompr_Level_Exposure,daioe_lngmod_Level_Exposure,daioe_translat_Level_Exposure,daioe_speechrec_Level_Exposure,daioe_genai_Level_Exposure
i64,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
2024,"""SSYK4""","""9629""","""Other service workers not else…","""women""","""60-64""","""Senior (50+)""",2159,30744,2159,131,275,426,6.459566,14.596603,24.58165,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,"""SSYK4""",33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,33.882353,28.705882,79.529412,42.352941,36.0,27.058824,21.647059,24.0,25.647059,27.294118,23.764706,2,2,4,3,2,2,2,2,2,2,2
2024,"""SSYK4""","""9629""","""Other service workers not else…","""men""","""60-64""","""Senior (50+)""",2640,30744,2640,43,222,350,1.655757,9.181141,15.283843,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,"""SSYK4""",33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,33.882353,28.705882,79.529412,42.352941,36.0,27.058824,21.647059,24.0,25.647059,27.294118,23.764706,2,2,4,3,2,2,2,2,2,2,2
2024,"""SSYK4""","""9629""","""Other service workers not else…","""women""","""55-59""","""Senior (50+)""",2067,30744,2067,-101,-70,-71,-4.658672,-3.27562,-3.320861,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,"""SSYK4""",33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,33.882353,28.705882,79.529412,42.352941,36.0,27.058824,21.647059,24.0,25.647059,27.294118,23.764706,2,2,4,3,2,2,2,2,2,2,2
2024,"""SSYK4""","""9629""","""Other service workers not else…","""men""","""55-59""","""Senior (50+)""",2437,30744,2437,-6,-101,-72,-0.2456,-3.979511,-2.869669,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,"""SSYK4""",33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,33.882353,28.705882,79.52

### 3.2 Export Processed Data

Write the reordered LazyFrame to parquet for downstream app use.

In [13]:
processed_lf_reorder_cols.sink_parquet(processed_data)